***# Load all 4 datasets***

In [2]:
import pandas as pd
import numpy as np

# Load all 4 datasets
crop_df  = pd.read_csv('Crop_Production_Data.csv')
rain_df  = pd.read_csv('rainfall_data_1901to2017.csv')
temp_df  = pd.read_csv('temperature.csv')
water_df = pd.read_csv('Indian_water_data.csv')

print("✓ All 4 datasets loaded")
print(f"  Crop:        {crop_df.shape}")
print(f"  Rainfall:    {rain_df.shape}")
print(f"  Temperature: {temp_df.shape}")
print(f"  Water:       {water_df.shape}")

✓ All 4 datasets loaded
  Crop:        (345336, 8)
  Rainfall:    (4188, 19)
  Temperature: (33, 13)
  Water:       (194, 23)


***# Fixing Column mane spaces***

In [5]:
# Strip whitespace from all column name in one line
crop_df.columns = crop_df.columns.str.strip()

# Verify the fix
print("Crop column after fix:")
print(crop_df.columns.tolist())

Crop column after fix:
['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield']


***# Fix the temperature column name***

In [6]:
# Rename the unnamed state column to something meaningful
temp_df = temp_df.rename(columns={'Unnamed: 0': 'State'})

# Verify the fix
print("Temperature column after fix:")
print(temp_df.columns.tolist())
print(temp_df.head(3))

Temperature column after fix:
['State', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'June', 'July', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
       State    Jan    Feb    Mar    Apr    May   June   July    Aug    Sep  \
0      bihar  18.65  22.88  30.00  36.25  38.52  37.44  33.02  32.33  30.98   
1  meghalaya  13.04  15.79  18.93  20.63  21.53  22.30  22.13  22.27  21.64   
2     punjab  14.09  17.69  24.01  32.07  37.67  39.46  36.28  33.04  31.69   

     Oct    Nov    Dec  
0  28.13  25.02  20.85  
1  19.66  16.77  14.00  
2  29.33  23.25  17.20  


***# Standardise all the state names (most important step)***

In [7]:
# --- Helper Function ---
# We'll apply this to every dataset
def clean_state_name(name):
  if pd.isna(name):
    return name
  name = str(name).strip()       # remove leading/trailling space
  name = name.title()            # Title Case: 'PUNJAB' -> 'Punjab'
  # Fix known spelling mistakes
  corrections = {
      'Andaman And Nicobar Island':  'Andaman And Nicobar Islands',
        'Andhra Padesh':               'Andhra Pradesh',
        'Rajastan':                    'Rajasthan',
        'Orissa':                      'Odisha',
        'The Dadra And Nagar Haveli':  'Dadra And Nagar Haveli',
        'Chandigarh':                  'Chandigarh',
  }
  return corrections.get(name, name)

# Apply to all datasets
crop_df['State']      = crop_df['State'].apply(clean_state_name)
temp_df['State']      = temp_df['State'].apply(clean_state_name)
water_df['State Name'] = water_df['State Name'].apply(clean_state_name)

# Verify
print("Crop states after cleaning:")
print(sorted(crop_df['State'].unique()))
print("Temperature states after cleaning:")
print(sorted(temp_df['State'].unique()))
print("Water states after cleaning:")
print(sorted(water_df['State Name'].unique()))

Crop states after cleaning:
['Andaman And Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra And Nagar Haveli', 'Daman And Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Laddak', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
Temperature states after cleaning:
['Andaman And Nicobarislands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra And Nagar Haveli', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Ut

***# Drop the 9 rows with missing Crop name***

In [8]:
print("Rows before:", len(crop_df))

crop_df = crop_df.dropna(subset=['Crop'])

print("Rows after dropping missing Crop:", len(crop_df))
print("Rows removed:", 345336 - len(crop_df))

Rows before: 345336
Rows after dropping missing Crop: 345327
Rows removed: 9


***# Handle missing Production values (Droping them)***

In [11]:
print("Rows berfore:", len(crop_df))
print("Missing production values:", crop_df['Production'].isna().sum())

# Dropping rows where Production values are missing
crop_df = crop_df.dropna(subset=['Production'])

print("Rows after dropping missing Production:", len(crop_df))
print("Rows removed:", 345327 - len(crop_df))

Rows berfore: 340383
Missing production values: 0
Rows after dropping missing Production: 340383
Rows removed: 4944


In [12]:
# Check current value
print("Before fix:")
print(temp_df[temp_df['State'].str.contains('Goa', na=False)][['State','July']])

Before fix:
  State   July
4   Goa  26.71


In [13]:
print("\nAfter fix:")
print(temp_df[temp_df['State'].str.contains('Goa', na=False)][['State','July']])
print("\nJuly column dtype:", temp_df['July'].dtype)


After fix:
  State   July
4   Goa  26.71

July column dtype: float64


***# Hnadle Rainfall missing values***

In [14]:
# Get all monthly + seasonal columns
rain_fill_cols = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG',
                  'SEP','OCT','NOV','DEC','ANNUAL','JF','MAM','JJAS','OND']

# Fill missing values with the MEDIAN of that column
# (groupby subdivision would be ideal but onverkill for <1% missing)
for col in rain_fill_cols:
  median_val = rain_df[col].median()
  rain_df[col] = rain_df[col].fillna(median_val)

# Varify
print("RAinfall missing values after fill")
print(rain_df[rain_fill_cols].isnull().sum())

RAinfall missing values after fill
JAN       0
FEB       0
MAR       0
APR       0
MAY       0
JUN       0
JUL       0
AUG       0
SEP       0
OCT       0
NOV       0
DEC       0
ANNUAL    0
JF        0
MAM       0
JJAS      0
OND       0
dtype: int64


***# Clean the water dataset, Drop the two junk columns and fill the rest***

In [16]:
# Step 1: Drop fecal-min and fecal-max (32% and 52% missing - unusable)
water_df = water_df.drop(columns=['Fecal - Min', 'Fecal - Max'])
print("Water columns after dropping junk:", water_df.shape[1], "columns remain")

# Step 2: Fill remaining missing values with column median
water_num_cols = water_df.select_dtypes(include='number').columns
for col in water_num_cols:
  water_df[col] = water_df[col].fillna(water_df[col].median())

# Varify
print("\nWater missing values after fill")
print(water_df.isnull().sum())

Water columns after dropping junk: 21 columns remain

Water missing values after fill
STN code                             0
Monitoring Location                  0
Year                                 0
Type Water Body                      0
State Name                           0
Temperature (C) - Min                0
Temperature (C) - Max                0
Dissolved - Min                      1
Dissolved - Max                      0
pH - Min                             0
pH - Max                             0
Conductivity (¬µmho/cm) - Min       13
Conductivity (¬µmho/cm) - Max        0
BOD (mg/L) - Min                    13
BOD (mg/L) - Max                     0
NitrateN (mg/L) - Min               13
NitrateN (mg/L) - Max                0
Fecal Coliform (MPN/100ml) - Min    13
Fecal Coliform (MPN/100ml) - Max    13
Total Coliform (MPN/100ml) - Min    14
Total Coliform (MPN/100ml) - Max     0
dtype: int64


***# IOR (Interquartile Range):***

In [17]:
print("Yield stats BEFORE outlier removal:")
print(crop_df['Yield'].describe())
print()

# Calculate IQR fences
Q1  = crop_df['Yield'].quantile(0.25)
Q3  = crop_df['Yield'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:.2f}")
print(f"Q3 = {Q3:.2f}")
print(f"IQR = {IQR:.2f}")
print(f"Lower fence = {lower_fence:.2f}")
print(f"Upper fence = {upper_fence:.2f}")
print()

# Keep only rows within the fences
before = len(crop_df)
crop_df = crop_df[(crop_df['Yield'] >= lower_fence) &
                  (crop_df['Yield'] <= upper_fence)]
after  = len(crop_df)

print(f"Rows removed as outliers: {before - after}")
print(f"Rows remaining: {after}")
print()
print("Yield stats AFTER outlier removal:")
print(crop_df['Yield'].describe())

Yield stats BEFORE outlier removal:
count    340383.000000
mean         80.578840
std         923.273309
min           0.000000
25%           0.570000
50%           1.030000
75%           2.500000
max       43958.330000
Name: Yield, dtype: float64

Q1 = 0.57
Q3 = 2.50
IQR = 1.93
Lower fence = -2.33
Upper fence = 5.40

Rows removed as outliers: 51392
Rows remaining: 288991

Yield stats AFTER outlier removal:
count    288991.000000
mean          1.218492
std           1.027953
min           0.000000
25%           0.500000
50%           0.900000
75%           1.580000
max           5.390000
Name: Yield, dtype: float64


***# Fearture Engineering on Temperature (Creat the 5 smart feature)***

In [18]:
# Monthly columns
month_cols = ['Jan','Feb','Mar','Apr','May','June','July','Aug','Sep','Oct','Nov','Dec']

# 1. Annual mean temperature
temp_df['annual_mean_temp'] = temp_df[month_cols].mean(axis=1)

# 2. Summer peak (Mar, Apr, May) — heat stress for crops
temp_df['summer_temp'] = temp_df[['Mar','Apr','May']].mean(axis=1)

# 3. Monsoon temp (Jun, Jul, Aug, Sep) — Kharif growing season
temp_df['monsoon_temp'] = temp_df[['June','July','Aug','Sep']].mean(axis=1)

# 4. Winter temp (Oct, Nov, Dec, Jan) — Rabi / wheat season
temp_df['winter_temp'] = temp_df[['Oct','Nov','Dec','Jan']].mean(axis=1)

# 5. Temperature range (hottest month - coldest month)
temp_df['temp_range'] = temp_df[month_cols].max(axis=1) - temp_df[month_cols].min(axis=1)

print("New temperature features:")
print(temp_df[['State','annual_mean_temp','summer_temp',
               'monsoon_temp','winter_temp','temp_range']].to_string())

New temperature features:
                         State  annual_mean_temp  summer_temp  monsoon_temp  winter_temp  temp_range
0                        Bihar         29.505833    34.923333       33.4425      23.1625       19.87
1                    Meghalaya         19.057500    20.363333       22.0850      15.8675        9.26
2                       Punjab         27.981667    31.250000       35.1175      20.9675       25.37
3               Madhya Pradesh         28.708333    35.343333       30.1650      23.2675       19.01
4                          Goa         28.495000    30.303333       27.1650      28.3825        4.21
5                    Rajasthan         29.000000    33.350000       33.5300      23.0325       20.11
6                    Telangana         28.895833    35.010000       28.4625      25.0450       13.77
7                      Gujarat         29.951667    33.763333       30.4375      27.5150       12.28
8       Dadra And Nagar Haveli         28.151667    30.063333    

***# Re-finxing the remaining missing values in water dataset:***

In [20]:
# Check exactly which columns still have missing values
print("Water columns with missing values:")
print(water_df.isnull().sum()[water_df.isnull().sum() > 0])
print()

# Fill any remaining numeric missing values
for col in water_df.columns:
    if water_df[col].isnull().sum() > 0:
        if water_df[col].dtype in ['float64', 'int64']:
            water_df[col] = water_df[col].fillna(water_df[col].median())
        else:
            water_df[col] = water_df[col].fillna(water_df[col].mode()[0])

# Verify
print("Missing values after fix:", water_df.isnull().sum().sum())
print("Status: ✓ CLEAN" if water_df.isnull().sum().sum() == 0 else "Still has issues")

Water columns with missing values:
Dissolved - Min                      1
Conductivity (¬µmho/cm) - Min       13
BOD (mg/L) - Min                    13
NitrateN (mg/L) - Min               13
Fecal Coliform (MPN/100ml) - Min    13
Fecal Coliform (MPN/100ml) - Max    13
Total Coliform (MPN/100ml) - Min    14
dtype: int64

Missing values after fix: 0
Status: ✓ CLEAN


***# Final health check on all datasets:***

In [21]:
print("=" * 50)
print("FINAL HEALTH CHECK — ALL DATASETS")
print("=" * 50)

for name, df in [("Crop", crop_df), ("Rainfall", rain_df),
                 ("Temperature", temp_df), ("Water", water_df)]:
    total_missing = df.isnull().sum().sum()
    print(f"\n{name}:")
    print(f"  Shape:          {df.shape}")
    print(f"  Total missing:  {total_missing}")
    print(f"  Status:         {'✓ CLEAN' if total_missing == 0 else '✗ Still has missing values'}")

FINAL HEALTH CHECK — ALL DATASETS

Crop:
  Shape:          (288991, 8)
  Total missing:  0
  Status:         ✓ CLEAN

Rainfall:
  Shape:          (4188, 19)
  Total missing:  0
  Status:         ✓ CLEAN

Temperature:
  Shape:          (33, 18)
  Total missing:  0
  Status:         ✓ CLEAN

Water:
  Shape:          (194, 21)
  Total missing:  0
  Status:         ✓ CLEAN


***# Save cleand datasets***

In [22]:
crop_df.to_csv('crop_clean.csv',  index=False)
rain_df.to_csv('rain_clean.csv',  index=False)
temp_df.to_csv('temp_clean.csv',  index=False)
water_df.to_csv('water_clean.csv', index=False)

print("✓ All 4 cleaned datasets saved!")
print("Files ready: crop_clean.csv, rain_clean.csv, temp_clean.csv, water_clean.csv")

✓ All 4 cleaned datasets saved!
Files ready: crop_clean.csv, rain_clean.csv, temp_clean.csv, water_clean.csv
